# 08 — Diseño de validación (Fase C)

Maquinaria de validación evaluada solo con las líneas base: prevalencias de entrada y permanencia, folds
walk-forward dentro de train, pérdida de la sensibilidad agrupada y métricas de las líneas base en cada fold.
No se entrena ningún modelo de ML y **el holdout no se abre**: los pares salen de
`validacion.cargar_pares_modelado` (solo train, criterio principal `imputacion_excluye`). La única vista del
holdout son conteos y tasas agregadas de `validacion.resumen_todas_particiones` (diagnóstico de deriva).

Definiciones (objetivos, folds, métricas, umbrales preregistrados) en `.claude/decisiones.md`, sección
"Objetivos de modelado y validación (Fase C)". Salidas en `data/meta/validacion_*.csv`.

In [ ]:
import json
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd

import constantes as cte
import pipeline_meta as pm
import universo_entrenamiento as ue
import validacion as va

TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
FEATURES = REPO / "data" / "06_features"
META = REPO / "data" / "meta"
COMMIT_GIT = pm.obtener_commit_git(REPO)
HORIZONTES = [1, 4]
N_BOOT = 200
SEED = 42
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)
t0_proceso = time.time()

## Prevalencias de entrada y permanencia

Clase positiva = `pobre_th == True`. Tabla principal con `imputacion_excluye`; `imputacion_conserva` como segunda
tabla. Tasas en %.

In [ ]:
resumenes = {h: va.resumen_todas_particiones(FEATURES, TRAYECTORIA, h) for h in HORIZONTES}
prevalencias = pd.concat([r["prevalencias"] for r in resumenes.values()], ignore_index=True)
prevalencias_anio = pd.concat([r["prevalencias_anio"] for r in resumenes.values()], ignore_index=True)
prevalencias.to_csv(META / "validacion_prevalencias.csv", index=False)
prevalencias_anio.to_csv(META / "validacion_prevalencias_anio.csv", index=False)


def tabla_prevalencias(criterio: str) -> pd.DataFrame:
    sub = prevalencias[prevalencias["criterio_imputacion"] == criterio].assign(
        tasa_positiva=lambda d: (100 * d["tasa_positiva"]).round(1))
    return sub.pivot_table(index=["objetivo_modelado", "universo", "particion", "era"], columns="h",
                           values=["n", "tasa_positiva"], aggfunc="first")


display(tabla_prevalencias("imputacion_excluye"))
display(tabla_prevalencias("imputacion_conserva"))

Por año de origen (universo 2003–2025, criterio principal, todas las particiones):

In [ ]:
anual = prevalencias_anio[(prevalencias_anio["universo"] == "universo_2003_2025")
                          & (prevalencias_anio["criterio_imputacion"] == "imputacion_excluye")]
anual = anual.groupby(["objetivo_modelado", "h", "ANO4_t"]).apply(
    lambda g: pd.Series({"n": g["n"].sum(), "tasa": 100 * np.average(g["tasa_positiva"], weights=g["n"]),
                         "particion": "+".join(sorted(g["particion"].unique()))}), include_groups=False).reset_index()
display(anual.pivot_table(index="ANO4_t", columns=["objetivo_modelado", "h"], values=["n", "tasa"],
                          aggfunc="first").round(1))

## Folds walk-forward dentro de train

Ventana expansiva, test = año de origen, embargo de h trimestres, mínimo `va.MIN_TRIMESTRES_TRAIN` trimestres de
origen en train. Los conteos de train son del criterio principal; los de test, del conjunto limpio (el mismo).

In [ ]:
pares = {h: va.cargar_pares_modelado(FEATURES, TRAYECTORIA, h) for h in HORIZONTES}
for h, p in pares.items():
    va.exigir_sin_holdout(p)
    print(f"h={h}: {len(p):,} pares de train, origen máximo {p['ANO4_t'].max()}T"
          f"{p.loc[p['ANO4_t'] == p['ANO4_t'].max(), 'TRIMESTRE_t'].max()}, particiones {sorted(p['particion'].unique())}")

pares_universo = {(u, h): p[ue.en_universo(p, u)] for h, p in pares.items() for u in cte.UNIVERSOS}
folds = {clave: va.generar_folds(p, clave[1]) for clave, p in pares_universo.items()}
tabla_folds = pd.concat([f.assign(universo=u) for (u, h), f in folds.items()], ignore_index=True)
tabla_folds.to_csv(META / "validacion_folds.csv", index=False)
display(tabla_folds.groupby(["universo", "h"])["anio_test"].agg(n_folds="size", desde="min", hasta="max"))
display(tabla_folds.set_index(["universo", "h", "anio_test"]))

## Sensibilidad agrupada: pares de test que se pierden

`evaluacion_sin_viviendas_train` saca de cada test las viviendas (`CODUSU`, `AGLOMERADO_t`) presentes en su train.

In [ ]:
perdida = pd.concat(
    [va.perdida_agrupada_folds(p, folds[(u, h)]).assign(universo=u, h=h) for (u, h), p in pares_universo.items()]
    + [r["perdida_agrupada_holdout"] for r in resumenes.values()], ignore_index=True)
perdida["pct_perdido"] = (100 * (1 - perdida["n_test_sin_viviendas_train"] / perdida["n_test"])).round(1)
perdida.to_csv(META / "validacion_perdida_agrupada.csv", index=False)
display(perdida.pivot_table(index="anio_test", columns=["universo", "h"], values="pct_perdido", aggfunc="first"))

## Líneas base en cada fold

Persistencia, media/tasa de train y decil de `ratio_linea_pobreza_t` (estimado en el train de la subpoblación),
sin ponderar y ponderadas. IC95% por bootstrap de vivienda solo para el decil. En h=1 se agrega la evaluación
agrupada (sin IC). La trayectoria también se evalúa sobre `variacion_log` (sin IC).

In [ ]:
t0 = time.time()
evaluaciones = []
for (universo, h), p in pares_universo.items():
    f = folds[(universo, h)]
    for objetivo in cte.OBJETIVOS_MODELADO:
        evaluaciones.append(va.evaluar_lineas_base(p, f, objetivo, n_boot=N_BOOT, seed=SEED).assign(universo=universo, h=h))
        if h == 1:
            evaluaciones.append(va.evaluar_lineas_base(p, f, objetivo, n_boot=0, agrupada=True).assign(universo=universo, h=h))
    evaluaciones.append(va.evaluar_lineas_base(p, f, "trayectoria", variable="log", n_boot=0).assign(universo=universo, h=h))
    print(f"{universo} h={h}: {time.time() - t0:.0f}s")
lineas_base = pd.concat(evaluaciones, ignore_index=True)
lineas_base = lineas_base.merge(tabla_folds[["universo", "h", "anio_test", "contexto_fold", "destino_en_contexto"]],
                                on=["universo", "h", "anio_test"], how="left")
lineas_base.to_csv(META / "validacion_lineas_base_folds.csv", index=False)
print(f"{len(lineas_base):,} filas")

### Trayectoria: skill del decil contra la persistencia y de las otras bases contra el decil

`skill_vs_base` = 1 − MSE / MSE del decil (el decil da 0 contra sí mismo); `skill_vs_persistencia` del decil
muestra cuánto le gana la referencia exigente a predecir variación 0. Sin ponderar, evaluación completa, simétrica.

In [ ]:
def vista(objetivo: str, metricas: list[str], linea_base: str = "decil_ratio", variable: str = "simetrica",
          ponderacion: str = "sin_ponderar", evaluacion: str = "evaluacion_completa") -> pd.DataFrame:
    sub = lineas_base[(lineas_base["objetivo_modelado"] == objetivo) & (lineas_base["linea_base"] == linea_base)
                      & (lineas_base["variable"] == variable) & (lineas_base["ponderacion"] == ponderacion)
                      & (lineas_base["evaluacion_agrupada"] == evaluacion) & lineas_base["metrica"].isin(metricas)]
    return sub.pivot_table(index=["universo", "h", "anio_test", "contexto_fold"], columns="metrica",
                           values="valor", aggfunc="first")[metricas]


trayectoria = vista("trayectoria", ["n", "r2_vs_media_train", "skill_vs_persistencia"]).join(
    vista("trayectoria", ["skill_vs_base"], linea_base="media_train").rename(columns={"skill_vs_base": "media_vs_decil"})).join(
    vista("trayectoria", ["skill_vs_base"], linea_base="persistencia").rename(columns={"skill_vs_base": "persistencia_vs_decil"}))
display(trayectoria.round(3))

### Entrada y permanencia: línea base por decil

Brier skill contra la tasa de train (la referencia climatológica; contra sí mismo el decil da 0), AUC-ROC, AUC-PR
relativa y calibración en grande: prevalencia vs. media predicha, intercepto y pendiente de calibración, ECE.

In [ ]:
METRICAS_CLASIFICACION = ["n", "prevalencia", "media_predicha", "brier_skill_vs_tasa_train", "auc_roc",
                          "auc_pr_relativa", "intercepto_calibracion", "pendiente_calibracion", "ece", "log_loss"]
for objetivo in ("entrada", "permanencia"):
    print(objetivo)
    display(vista(objetivo, METRICAS_CLASIFICACION).round(3))

### IC95% del decil (bootstrap por vivienda) y lectura ponderada

Rango de los IC del skill principal y diferencia entre la lectura sin ponderar y la ponderada, por objetivo.

In [ ]:
METRICA_PRINCIPAL = {"trayectoria": "skill_vs_persistencia", "entrada": "brier_skill_vs_tasa_train",
                     "permanencia": "brier_skill_vs_tasa_train"}
filas = []
for objetivo, metrica in METRICA_PRINCIPAL.items():
    sub = lineas_base[(lineas_base["objetivo_modelado"] == objetivo) & (lineas_base["linea_base"] == "decil_ratio")
                      & (lineas_base["metrica"] == metrica) & (lineas_base["variable"] == "simetrica")
                      & (lineas_base["evaluacion_agrupada"] == "evaluacion_completa")]
    ancho = (sub["ic95_high"] - sub["ic95_low"])
    filas.append(sub.assign(ancho_ic=ancho).groupby(["universo", "h", "ponderacion"]).agg(
        valor_medio=("valor", "mean"), valor_min=("valor", "min"), valor_max=("valor", "max"),
        ancho_ic_medio=("ancho_ic", "mean")).assign(objetivo_modelado=objetivo, metrica=metrica))
display(pd.concat(filas).reset_index().set_index(["objetivo_modelado", "universo", "h", "ponderacion"]).round(3))

### Evaluación agrupada (h=1): cambio del skill del decil al excluir las viviendas de train

In [ ]:
completa = vista("trayectoria", ["skill_vs_persistencia"])
agrupada = vista("trayectoria", ["skill_vs_persistencia"], evaluacion="evaluacion_sin_viviendas_train")
comparacion = completa.join(agrupada, rsuffix="_agrupada", how="inner")
for objetivo in ("entrada", "permanencia"):
    c = vista(objetivo, ["brier_skill_vs_tasa_train", "auc_roc"])
    a = vista(objetivo, ["brier_skill_vs_tasa_train", "auc_roc"], evaluacion="evaluacion_sin_viviendas_train")
    comparacion = comparacion.join(c.join(a, rsuffix="_agrupada").add_prefix(f"{objetivo}_"), how="left")
display(comparacion.round(3))

## Candado: sin aperturas del holdout y registro de la corrida

In [ ]:
corridas = pd.read_csv(META / "corridas.csv", dtype=str)
aperturas = [json.loads(p).get("evento") == va.EVENTO_APERTURA for p in corridas["parametros_json"].dropna()]
print(f"eventos {va.EVENTO_APERTURA} en corridas.csv: {sum(aperturas)}")
assert sum(aperturas) == 0

duracion = time.time() - t0_proceso
pm.registrar_corrida(
    META / "corridas.csv", "08_diseno_validacion",
    {"HORIZONTES": HORIZONTES, "N_BOOT": N_BOOT, "SEED": SEED, "criterio_imputacion": cte.CRITERIO_IMPUTACION_PRINCIPAL,
     "min_trimestres_train": va.MIN_TRIMESTRES_TRAIN, "holdout_desde": va.HOLDOUT_DESDE},
    int(tabla_folds.groupby(["universo", "h"]).size().sum()), len(lineas_base), duracion, COMMIT_GIT)
print(f"corrida registrada: {len(tabla_folds)} folds, {len(lineas_base):,} filas, {duracion:.0f}s")